# Open Design on Jute — Host-Shell Design Spec

**Date:** 2026-05-31  ·  **Status:** Draft for review  ·  **Authors:** brain (Claude) + Kevin

> Use the **Jute notebook** as the visualization layer for **Open Design (OD)**.
> An OD design *project* becomes a Jute *notebook*; the SPUR brain agent *operates*
> that notebook through `notebook_*` MCP tools; each design *artifact* is a *cell*
> whose output renders in Jute's sandboxed iframe.

This spec is itself authored as a SPUR notebook — markdown + Mermaid diagrams +
live HTML/JS mockups rendered as cell outputs — so it doubles as a working
demonstration of the thing it proposes.


## Why this exists & the core reframe

OD is the open-source **Claude Design** alternative: a local-first product where a
coding-agent CLI behaves like a senior designer and ships *artifacts* (decks,
landing pages, posters) instead of prose. Today OD's **visualization layer** is a
React web app — a chat panel, a live todo card, a file workspace, and a
**sandboxed `srcdoc` iframe** that renders each `<artifact>`.

Jute is a native **Tauri + React** reimagining of the Jupyter frontend: *kernel as
a window*, a reactive DAG, a **deck mode** (cell↔slide), and an **agent bridge**.

**The pivotal realization:** Jute's `AgentBridge` is *inverted* from OD's daemon.
OD **spawns** a CLI and **parses** its `<artifact>` stream. Jute instead **exposes
the notebook as MCP tools** (`notebook_insert_cell`, `notebook_write_cell`,
`notebook_read_cell`, `notebook_set_cell_metadata`, snapshot…) and lets the SPUR
brain agent **drive** it.

So we do not port OD's agent-spawning machinery — **we delete it.** What we keep is
OD's *prompt stack* (the senior-designer behaviour: discovery → directions → todo →
critique → artifact) and its *design assets* (72 design systems, 31 skills, 5
directions, device frames). Those are data + prompt strings — they move into a SPUR
**"open-design" skill package** that teaches the brain agent how to drive a design
notebook.


## Locked decisions

| # | Decision | Choice |
|---|----------|--------|
| 1 | **What is the visualization layer?** | **Jute becomes the host shell** — the whole OD experience lives inside the Jute Tauri app. |
| 2 | **Runtime for OD's daemon logic** | **Reuse SPUR's Rust `AgentBridge`/ACP**; port only OD's *prompt stack* (no Node daemon shipped). |
| 3 | **Artifact → notebook mapping** | **Hybrid C** — artifact-as-`text/html`-output is the universal substrate; **deck mode** specializes `kind: deck`; the **reactive DAG** specializes live/data-bound artifacts. Build the substrate first. |

**Confirmed feasibility (read from the code, not assumed):**

- Jute's `OutputView.tsx` already renders `text/html` outputs in a **sandboxed iframe**
  (`sandbox='allow-scripts'`, no `allow-same-origin`, auto-height via the injected
  `jute-iframe-height` reporter). This *is* OD's artifact-preview pattern — already shipped.
- The `notebook_*` MCP tool surface (snapshot / read / insert / write / delete / set-metadata)
  already exists and is what this very notebook is being built through.
- Mermaid renders inside markdown cells (settings-gated). Deck mode (`src/ui/deck/`) and
  the reactive DAG engine already exist — the specializations are *wiring*, not new infra.


## System architecture

```mermaid
flowchart TB
  user([Designer]) -->|brief, redirects| shell

  subgraph shell["Jute host shell · Tauri + React"]
    nbview["Notebook view\n(cells = process + artifacts)"]
    deck["Deck mode\n(cell ↔ slide)"]
    dag["Reactive DAG\n(data → transform → render)"]
    outv["OutputView\nsandboxed iframe (text/html)"]
    nbview --> outv
    deck --> outv
    dag --> outv
  end

  subgraph rt["SPUR agent runtime · Rust"]
    bridge["AgentBridge / ACP"]
    tools["notebook_* MCP tools"]
    bridge --- tools
  end

  subgraph brain["SPUR brain agent (Claude Code / Codex / …)"]
    odskill["open-design skill package\nprompts · 72 design systems · 31 skills · 5 directions"]
  end

  shell <-->|agent://request / response| bridge
  brain -->|drives via tools| tools
  odskill -.injected.-> brain

  classDef gone stroke-dasharray:4 3,color:#999;
  noded["✗ OD Node daemon\n(spawn CLIs, parse &lt;artifact&gt; SSE, HTTP proxy)"]:::gone
  noded -.deleted.-> rt
```

The three layers map cleanly onto the three subsystems below.


## Subsystem mapping — OD today → Jute host shell

| Subsystem | OD today | In the Jute host shell |
|---|---|---|
| **Project / persistence** | SQLite `.od/app.sqlite` (projects, conversations, tabs, templates) | The **`.ipynb` file *is* the project** — brief, transcript, todos, and artifacts in one document. Jute's `notebook_store` + recents replace SQLite. |
| **Agent runtime** | Node daemon spawns 16 CLIs, normalizes SSE, parses `<artifact>` | **SPUR brain agent over ACP / `AgentBridge`**, operating the notebook via `notebook_*` tools. *No Node daemon.* |
| **Prompt stack** | `packages/contracts/src/prompts/*` fed to the spawned CLI | A **SPUR `open-design` skill** injected into the brain agent: discovery form, directions, skills, design-systems, critique. |
| **Visualization** | Web app: chat + iframe preview + todo card + file workspace | **Jute cells**: artifact-as-`text/html`-output (substrate) · deck mode (decks) · reactive DAG (live artifacts). |
| **Export** | daemon: HTML / PDF / PPTX / ZIP / MD | Notebook export + per-artifact export from the rendered cell (PDF/PPTX deferred to a later milestone). |


## The agent operating loop

Instead of OD's "spawn → stream → parse", the brain agent **edits the notebook**.
Each step is a cell it inserts or rewrites; the user sees it live and can redirect mid-flight.

```mermaid
sequenceDiagram
  participant U as Designer
  participant A as Brain agent
  participant N as Notebook (notebook_* tools)
  participant V as OutputView (iframe)

  U->>A: "magazine pitch deck for our seed round"
  A->>N: insert_cell(markdown, discovery form)
  U->>N: answers (surface / audience / tone / scale)
  A->>N: read_cell(answers)
  A->>N: insert_cell(markdown, direction picker — 5 schools)
  U->>N: picks "Editorial Monocle"
  A->>N: insert_cell(markdown, TodoWrite plan)
  loop per todo
    A->>N: write_cell(artifact cell · text/html)
    N->>V: render in sandboxed iframe
    A->>N: set_cell_metadata(status: completed)
  end
  A->>A: 5-dimensional self-critique
  A->>N: write_cell(artifact, revised)
  U-->>A: redirect (cheap, mid-flight)
```


## Artifact mapping — hybrid **C**

One host model serves every OD artifact kind. The substrate is built first; deck and
DAG are kind-specific upgrades that reuse infrastructure Jute already has.

```mermaid
flowchart LR
  art["Agent emits artifact\n(kind + entry HTML/JSX)"] --> k{kind?}
  k -->|deck| deckm["Deck mode\ncell ↔ slide\n(src/ui/deck)"]
  k -->|live / data-bound| dagm["Reactive DAG\ndata.json → transform → template"]
  k -->|html · jsx · svg · poster · mini-app| sub["Artifact cell\ntext/html output"]
  deckm --> iframe["Sandboxed iframe\n(allow-scripts)"]
  dagm --> iframe
  sub --> iframe

  classDef now fill:#e8f5e9,stroke:#43a047;
  classDef next fill:#fff8e1,stroke:#fb8c00;
  class sub,iframe now;
  class deckm,dagm next;
```

🟢 = build first (M1 substrate) 🟠 = layer on (M2 deck, M3 live-DAG).


## UI/UX mockup — the host shell

What the integrated product looks like: the agent panel on the left drives the
notebook on the right; the final artifact is a cell whose output renders in the
sandboxed iframe. (Rendered as a `text/html` cell output — the very mechanism this
spec proposes. Enable *active content* in Jute to exercise the direction-picker JS.)


In [2]:
# Host-shell UI mockup — rendered as a text/html cell output
# (agent panel + notebook + artifact cell in a sandboxed iframe)

## UI/UX mockup — an artifact cell, full-bleed

A single design artifact as it renders *inside one cell's output*. This is what the
agent produces with `write_cell` + a `text/html` payload. Decks swap this for
deck-mode slides; live artifacts swap the static body for a `data.json`-driven DAG render.


In [1]:
# Sample design artifact (deck slide) — what a single artifact cell renders

<rendered HTML artifact>

## Prompt-stack port plan

OD's "brain" is mostly *data + prompt strings*, so the port is re-homing, not rewriting.

| OD source | Lands as | Effort |
|---|---|---|
| `packages/contracts/src/prompts/system.ts`, `discovery.ts`, `directions.ts`, `deck-framework.ts` | The `open-design` SPUR skill's system/instruction text | Low — text move + notebook-tool framing |
| `skills/*` (31), `design-systems/*` (72), `assets/frames/*` | Bundled reference assets the skill points the agent at | Low — copy + index |
| 5-dimensional critique + anti-AI-slop checklist | A skill section + an explicit critique step in the loop | Low |
| Discovery / direction **forms** | Markdown cells with structured metadata the agent reads back (M1); interactive cell type (later) | Medium |
| `<artifact>` parsing + manifest | Replaced by `write_cell` + a cell-level **artifact manifest** in `metadata.spur.artifact` | Medium |
| Node daemon: `agents.ts`, `*-stream.ts`, `acp.ts`, MCP servers, HTTP proxy | **Deleted** — replaced by SPUR `AgentBridge`/ACP + `notebook_*` tools | n/a (removal) |
| `pdf-export.ts`, `pptx`, ZIP | Deferred to an export milestone | — |


## Milestones

**M1 — Vertical slice (substrate).** One skill (e.g. `saas-landing`), `kind: html`.
Brain agent runs the full loop via `notebook_*` tools; final artifact renders as a
`text/html` cell output. Notebook file is the project. *Exit:* end-to-end "brief →
rendered artifact in a cell" with no Node daemon.

**M2 — Deck specialization.** Route `kind: deck` into Jute's existing deck mode;
agent emits one cell per slide; present/export. *Exit:* a generated deck presents
in deck mode.

**M3 — Live-artifact DAG.** Map OD live-artifacts (`data.json → transform →
template`) onto Jute's reactive DAG so a data-bound design refreshes from a
source. *Exit:* a dashboard artifact re-renders on data change.

**M4 — Polish.** Direction/discovery as interactive cell types, export
(PDF/PPTX/ZIP), design-system browser, frames. *Exit:* parity with OD's core UX.


## Open questions & next steps

1. **Artifact manifest home** — confirm `metadata.spur.artifact` (kind, title,
   renderer, designSystemId) as the per-cell manifest; does `set_cell_metadata`
   carry arbitrary objects today?
2. **Skill injection mechanism** — is `open-design` a standard SPUR skill package,
   or does it need a dedicated agent profile?
3. **Forms in M1** — markdown-with-metadata is the cheap path; when do we invest in
   a real interactive cell type?
4. **Multi-file artifacts** — OD reserves `supportingFiles`; how do those live in a
   single `.ipynb` (attachments? sidecar dir)?
5. **Active-content trust** — design artifacts need scripts; per-notebook trust
   (Jupyter-style signature) is currently deferred in `OutputView`.

**Next:** confirm this reframe, then I'll turn M1 into an implementation plan
(writing-plans skill).
